# Pre Processing of a Emotions Video Dataset for a Multimodal Emotion Recognition

## Dataset

In [1]:
# imports

from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [2]:
ROOT = Path("../videos")

LABELS = {
    "Anger":0,
    "Disgust":1,
    "Fear":2,
    "Happy":3,
    "Neutral":4,
    "Sadness":5,
    "Surprised":6
}

In [3]:
videos = []

for emotion in LABELS:

    folder = ROOT / emotion

    for video in folder.glob("*.mp4"):

        videos.append({
            "video_path": str(video),
            "label": emotion,
            "label_id": LABELS[emotion]
        })

dataset = pd.DataFrame(videos)

In [4]:
dataset.sample(n=10)

,video_path,label,label_id
190,..\videos\Fear\fear_0022.mp4,Fear,2
482,..\videos\Sadness\sadness_0053.mp4,Sadness,5
390,..\videos\Neutral\neutral_0054.mp4,Neutral,4
567,..\videos\Surprised\surprised_0059.mp4,Surprised,6
16,..\videos\Anger\anger_0017.mp4,Anger,0
509,..\videos\Surprised\surprised_0001.mp4,Surprised,6
105,..\videos\Disgust\disgust_0022.mp4,Disgust,1
52,..\videos\Anger\anger_0053.mp4,Anger,0
86,..\videos\Disgust\disgust_0003.mp4,Disgust,1
500,..\videos\Sadness\sadness_0071.mp4,Sadness,5


In [5]:
dataset.groupby("label").size()

label
Anger        84
Disgust      85
Fear         83
Happy        85
Neutral      93
Sadness      79
Surprised    90
dtype: int64

In [6]:
dataset.to_csv(
    "../datasets/dataset.csv",
    index=False
)

## Audio Extraction (ffmpeg)

In [7]:
%pip install ffmpeg-python

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [8]:
import ffmpeg

def extract_audio(video_path, output_path):

    (
        ffmpeg
        .input(str(video_path))
        .output(
            str(output_path),
            ac=1, # mono channel
            ar=16000 # sample rate
        )
        .overwrite_output()
        .run()
    )

In [9]:
import os

os.makedirs("../processed/audio", exist_ok=True)

for row in dataset.itertuples():

    extract_audio(
        row.video_path,
        f"../processed/audio/{Path(row.video_path).stem}.wav"
    )

Áudios salvos em `.\processed\audio`

## Transcription (whisper)

In [10]:
%pip install transformers accelerate sentencepiece librosa soundfile

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:
from pathlib import Path

import torch

from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    pipeline
)

from tqdm import tqdm

C:\Users\heric\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(DEVICE)

cpu


In [13]:
# varia de acordo com o device
if DEVICE == "cuda":
    MODEL_NAME = "openai/whisper-large-v2"
else:
    MODEL_NAME = "openai/whisper-small"

In [14]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
)

model.to(DEVICE)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 479/479 [00:00<00:00, 2369.20it/s]


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [15]:
asr = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device=0 if DEVICE == "cuda" else -1,
)

In [16]:
TRANSCRIPT_DIR = Path("../processed/transcripts")

TRANSCRIPT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [17]:
import torch

# Carregar VAD e utilitários 
vad_model, vad_utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False,
    trust_repo=True 
)

(get_speech_timestamps, _, read_audio, _, collect_chunks) = vad_utils

Using cache found in C:\Users\heric/.cache\torch\hub\snakers4_silero-vad_master


In [18]:
import torch
import soundfile as sf

def load_audio_tensor(audio_path: str) -> torch.Tensor:
    """Carrega áudio .wav em float32 16kHz sem depender do torchaudio/torchcodec."""
    data, sr = sf.read(audio_path)
    tensor = torch.from_numpy(data).float()
    
    # Se por acaso for estéreo, converte para mono
    if tensor.ndim > 1:
        tensor = tensor.mean(dim=1)
        
    return tensor


def transcribe_audio(audio_path: str) -> str:
    # 1. Carrega o áudio via soundfile
    wav = load_audio_tensor(audio_path)
    
    # 2. Identifica os timestamps onde há fala
    speech_timestamps = get_speech_timestamps(
        wav, 
        vad_model, 
        sampling_rate=16000, 
        threshold=0.5
    )
    
    # 3. Se só houver ruído/silêncio, retorna string vazia
    if not speech_timestamps:
        return ""
        
    # 4. Concatena os trechos de fala
    speech_wav = collect_chunks(speech_timestamps, wav)
    
    # 5. Transcreve via pipeline 'asr' (com return_timestamps=True para aceitar áudios > 30s)
    result = asr(
        speech_wav.numpy(),
        return_timestamps=True, 
        generate_kwargs={
            "task": "transcribe",
            "language": "en",
            "temperature": 0.0,
        }
    )

    return result["text"].strip()

In [19]:
audio = "../processed/audio/anger_0004.wav"

texto = transcribe_audio(audio)

print(texto)

In [20]:
import os
from pathlib import Path
from tqdm import tqdm


os.makedirs("../processed/transcripts", exist_ok=True)

def process_and_save_transcriptions(df):
    for row in tqdm(df.itertuples(), total=len(df), desc="Gerando Transcrições"):
        # Extrai o nome base (ex: 'disgust_0051')
        sample_id = Path(row.video_path).stem 
        
        # Caminho do áudio gerado anteriormente e da saída do texto
        audio_path = f"../processed/audio/{sample_id}.wav"
        txt_path = f"../processed/transcripts/{sample_id}.txt"
        
        # Pula se já foi transcrito
        if os.path.exists(txt_path):
            continue
            
        # Garante que o áudio extraído existe
        if not os.path.exists(audio_path):
            print(f" [Aviso] Áudio não encontrado: {audio_path}")
            continue
            
        # Chama a função VAD + Whisper
        transcription = transcribe_audio(audio_path)
        
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(transcription)

In [21]:
# Processar transcrições
process_and_save_transcriptions(dataset)

Gerando Transcrições:   0%|          | 0/599 [00:00<?, ?it/s][transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensA

KeyboardInterrupt: 

## Video Sample (retinaface + videomae)

In [6]:
%pip install retina-face tf-keras

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import cv2
import torch
from tqdm import tqdm

# Força o TensorFlow a usar o Keras 2 (legado) em vez do Keras 3
os.environ["TF_USE_LEGACY_KERAS"] = "1"
from retinaface import RetinaFace 

os.makedirs("../processed/faces", exist_ok=True)

def extract_and_save_faces(video_path: str, output_dir: str, num_frames: int = 16):
    """
    Extrai num_frames uniformes do vídeo, detecta o rosto com RetinaFace,
    recorta e salva em JPG dentro de output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames <= 0:
        cap.release()
        return

    # Amostragem uniforme dos índices dos frames
    frame_indices = [int(i * total_frames / num_frames) for i in range(num_frames)]
    
    saved_count = 0
    current_frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or saved_count >= num_frames:
            break

        if current_frame_idx in frame_indices:
            # RetinaFace espera RGB
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Detecta faces
            faces = RetinaFace.detect_faces(rgb_frame)
            
            if isinstance(faces, dict) and len(faces) > 0:
                # Pega a primeira face detectada (maior confiança/chave 'face_1')
                face_info = faces['face_1']
                facial_area = face_info['facial_area'] # [x1, y1, x2, y2]
                
                x1, y1, x2, y2 = facial_area
                
                # Adiciona margem de segurança (padding) se desejar
                h, w, _ = frame.shape
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                
                cropped_face = frame[y1:y2, x1:x2]
                
                # Redimensiona para dimensão padrão (ex: 224x224 para VideoMAE)
                resized_face = cv2.resize(cropped_face, (224, 224))
                
                # Salva imagem do rosto
                img_path = os.path.join(output_dir, f"frame_{saved_count:02d}.jpg")
                cv2.imwrite(img_path, resized_face)
            else:
                # Fallback: Se não detectar face no frame específico, salva o frame original redimensionado
                resized_frame = cv2.resize(frame, (224, 224))
                img_path = os.path.join(output_dir, f"frame_{saved_count:02d}.jpg")
                cv2.imwrite(img_path, resized_frame)

            saved_count += 1

        current_frame_idx += 1

    cap.release()



def process_all_video_samples(df, num_frames=16):
    os.makedirs("../processed/faces", exist_ok=True)
    
    for row in tqdm(df.itertuples(), total=len(df), desc="Extraindo Rostos"):
        sample_id = Path(row.video_path).stem
        sample_faces_dir = f"../processed/faces/{sample_id}"
        
        # Pula se a pasta já tiver os frames processados
        if os.path.exists(sample_faces_dir) and len(os.listdir(sample_faces_dir)) == num_frames:
            continue
            
        extract_and_save_faces(row.video_path, sample_faces_dir, num_frames=num_frames)


In [6]:
process_all_video_samples(dataset, num_frames=8)

Extraindo Rostos: 100%|██████████| 599/599 [20:15:34<00:00, 121.76s/it]   


## Embeddings

In [23]:
import os
import glob
from pathlib import Path
import torch
import torch.nn as nn
from PIL import Image
import soundfile as sf
from tqdm import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModel,
    Wav2Vec2Processor, 
    Wav2Vec2Model,
    VideoMAEImageProcessor, 
    VideoMAEModel
)

# 1. Configurar Device e Pastas de Cache
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizando dispositivo: {device}")

dirs = [
    "../cache/text_embeddings",
    "../cache/audio_embeddings",
    "../cache/image_embeddings",
    "../cache/multimodal"
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# 2. Carregar Backbones e Processadores (Congelados)

# Text: BERT em inglês (transcrições já geradas em inglês pelo Whisper)
print("Carregando BERT (inglês)...")
text_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text_model = AutoModel.from_pretrained("bert-base-uncased").to(device).eval()

# Audio: Wav2Vec2
print("Carregando Wav2Vec2...")
audio_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
audio_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device).eval()

# Carregar a versão ajustada no Kinetics-400
print("Carregando VideoMAE (Kinetics-400)...")
video_processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
video_model = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics").to(device).eval()

# Congelar todos os parâmetros explicitamente
for m in [text_model, audio_model, video_model]:
    for p in m.parameters():
        p.requires_grad = False

Utilizando dispositivo: cpu
Carregando BERT (inglês)...


C:\Users\heric\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\heric\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3996.11it/s]
[transformers] BertModel LOAD REPORT fr

Carregando Wav2Vec2...


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 2281.66it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Carregando VideoMAE (Kinetics-400)...


Loading weights: 100%|██████████| 158/158 [00:00<00:00, 2646.22it/s]
[transformers] VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key                                                        | Status     | 
-----------------------------------------------------------+------------+-
videomae.encoder.layer.{0...11}.attention.attention.q_bias | UNEXPECTED | 
classifier.bias                                            | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.v_bias | UNEXPECTED | 
classifier.weight                                          | UNEXPECTED | 
fc_norm.bias                                               | UNEXPECTED | 
fc_norm.weight                                             | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.bias        | MISSING    | 
encoder.layer.{0...11}.attention.attention.value.bias      | MISSING    | 
encoder.layer.{0...11}.attention.attention.query.bias      | MISSING    | 

Notes:
- UNEXPECT

In [24]:
@torch.no_grad()
def get_text_embedding(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read().strip()
    
    if not text:
        text = "[SILÊNCIO]"
        
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    outputs = text_model(**inputs)
    
    # Mean pooling ao longo da sequência -> Shape: [768]
    text_emb = outputs.last_hidden_state.mean(dim=1).squeeze(0)
    return text_emb.cpu()


@torch.no_grad()
def get_audio_embedding(audio_path):
    speech, sr = sf.read(audio_path)
    inputs = audio_processor(speech, sampling_rate=sr, return_tensors="pt").to(device)
    outputs = audio_model(**inputs)
    
    # Mean pooling -> Shape: [768]
    audio_emb = outputs.last_hidden_state.mean(dim=1).squeeze(0)
    return audio_emb.cpu()


@torch.no_grad()
def get_image_embedding(faces_dir, target_frames=16):
    # Carrega todos os frames da pasta e ordena
    image_paths = sorted(glob.glob(os.path.join(faces_dir, "*.jpg")) + glob.glob(os.path.join(faces_dir, "*.png")))
    
    if len(image_paths) == 0:
        raise FileNotFoundError(f"Nenhum frame encontrado em: {faces_dir}")
        
    # Garante EXATAMENTE 16 frames
    num_found = len(image_paths)
    if num_found != target_frames:
        # Amostragem/Reaproveitamento de índices para forçar 16 frames
        indices = [int(i * num_found / target_frames) for i in range(target_frames)]
        image_paths = [image_paths[i] for i in indices]

    images = [Image.open(p).convert("RGB") for p in image_paths]
    
    # VideoMAE espera a lista com 16 frames
    inputs = video_processor(images, return_tensors="pt").to(device)
    outputs = video_model(**inputs)
    
    # Mean pooling -> Shape: [768]
    image_emb = outputs.last_hidden_state.mean(dim=1).squeeze(0)
    return image_emb.cpu()

## Cache

In [25]:
def generate_all_embeddings(df):
    for row in tqdm(df.itertuples(), total=len(df), desc="Gerando Cache Embeddings"):
        sample_id = Path(row.video_path).stem
        label_id = row.label_id
        
        # Caminhos dos artefatos salvos
        txt_path = f"../processed/transcripts/{sample_id}.txt"
        audio_path = f"../processed/audio/{sample_id}.wav"
        faces_dir = f"../processed/faces/{sample_id}"
        
        # Destinos no cache
        multimodal_cache_path = f"../cache/multimodal/{sample_id}.pt"
        
        # Pula se o cache multimodal já existir
        if os.path.exists(multimodal_cache_path):
            continue
            
        try:
            # 1. Extrair Embeddings
            text_emb = get_text_embedding(txt_path)
            audio_emb = get_audio_embedding(audio_path)
            image_emb = get_image_embedding(faces_dir)
            
            # 2. Salvar individualmente nas pastas de modalidades
            torch.save(text_emb, f"../cache/text_embeddings/{sample_id}.pt")
            torch.save(audio_emb, f"../cache/audio_embeddings/{sample_id}.pt")
            torch.save(image_emb, f"../cache/image_embeddings/{sample_id}.pt")
            
            # 3. Salvar o dicionário consolidado para o treino
            sample_data = {
                "text": text_emb,      # Tensor [768]
                "audio": audio_emb,    # Tensor [768]
                "image": image_emb,    # Tensor [768]
                "label": torch.tensor(label_id, dtype=torch.long)
            }
            torch.save(sample_data, multimodal_cache_path)
            
        except Exception as e:
            print(f"\n[Erro] Falha ao processar {sample_id}: {e}")

In [26]:
# Executa para o dataset
generate_all_embeddings(dataset)

Gerando Cache Embeddings: 100%|██████████| 599/599 [58:10<00:00,  5.83s/it]  
